# Crypto market microstructure: four questions, answered with baselines

**Data:** 911k one-minute candles for 8 Coinbase pairs over 90 days, from the dbt marts in `analytics/` (pipeline candles where the stream ran, official Coinbase candles otherwise).

**Method:** every estimate comes with a 95% confidence interval and a baseline or null hypothesis. Minutes within a day are not independent, so CIs resample **whole days** (block bootstrap). Random draws use a fixed seed, so `make analysis` reproduces every number.

> Descriptive market statistics for a data-engineering portfolio project. Not trading or investment advice.

| # | Question | Mart |
|---|---|---|
| Q1 | Do volatile hours follow volatile hours? | `mart_volatility_hourly` |
| Q2 | When in the day and week is each market most active? | `mart_volatility_hourly` |
| Q3 | After an extreme 1-minute move, does the price continue or revert? | `mart_minute_outcomes` |
| Q4 | How closely does the streaming pipeline agree with the exchange? | `mart_pipeline_vs_exchange_summary` |

In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
from scipy import stats as sps

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
sys.path.insert(0, str(ROOT))
from analysis.stats import acf, block_bootstrap_ci, holm, shuffle_null_band, two_proportion_z, wilson_ci  # noqa: E402
from src.config import POSTGRES_CONNECT_KWARGS  # noqa: E402  (reads .env)

warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
SEED = 20260927
CHARTS = ROOT / "analysis" / "charts"
CHARTS.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

conn = psycopg2.connect(**POSTGRES_CONNECT_KWARGS)


def q(sql):
    return pd.read_sql(sql, conn)


def rng():
    return np.random.default_rng(SEED)


findings = []  # one row per headline number, printed at the end


def fmt_ci(ci, digits=3):
    return f"[{ci[0]:.{digits}f}, {ci[1]:.{digits}f}]"


q("select count(*) as candles, count(distinct crypto_id) as pairs, min(bucket)::date as first_day, max(bucket)::date as last_day from analytics.fct_candles_1m")

,candles,pairs,first_day,last_day
0,911894,8,2026-06-29,2026-09-27


## Q1. Do volatile hours follow volatile hours?

**Measure:** hourly realized volatility, √Σr² of the 1-minute log returns, for hours with at least 30 valid returns.

**Deseasonalize first:** every market is busier at some UTC hours than others, which by itself would make neighbouring hours look alike. Each hour is divided by that pair's mean for the same UTC hour, then logged.

**Test:** the autocorrelation at lags 1–24 hours, compared with a **null band** from 1,000 random shuffles of the hours (no time structure).

**Uncertainty:** the lag-1 CI comes from resampling whole days of (hour, next hour) pairs.

In [2]:
vol = q("select symbol, hour_start, realized_vol, n_returns, volume from analytics.mart_volatility_hourly")
vol["hour_start"] = pd.to_datetime(vol.hour_start, utc=True)
vol = vol[vol.n_returns >= 30].copy()
vol["hour_utc"] = vol.hour_start.dt.hour
vol["weekday"] = vol.hour_start.dt.dayofweek + 1  # ISO: 1 = Monday
vol["deseason"] = np.log(vol.realized_vol / vol.groupby(["symbol", "hour_utc"]).realized_vol.transform("mean"))

LAGS = 24
q1 = []
fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharey=True)
for ax, (sym, g) in zip(axes.flat, vol.groupby("symbol")):
    series = g.set_index("hour_start").deseason.sort_index()
    grid = series.reindex(pd.date_range(series.index.min(), series.index.max(), freq="h"))  # NaN = missing hour
    x = grid.to_numpy()
    rho = acf(x, LAGS)
    lo, hi = shuffle_null_band(x, LAGS, n=1000, rng=rng())

    # lag-1 CI: resample whole days of (hour, next hour) pairs
    pairs = pd.DataFrame({"a": x[:-1], "b": x[1:], "day": grid.index[:-1].date}).dropna()
    blocks = [d[["a", "b"]].to_numpy() for _, d in pairs.groupby("day")]
    ci = block_bootstrap_ci(blocks, lambda rows: np.corrcoef(rows[:, 0], rows[:, 1])[0, 1], rng=rng())
    q1.append({"symbol": sym, "lag1": rho[0], "ci_lo": ci[0], "ci_hi": ci[1], "null_hi": hi[0],
               "beyond_null": rho[0] > hi[0], "lag24": rho[23], "hours": int(np.isfinite(x).sum())})

    lags = np.arange(1, LAGS + 1)
    ax.fill_between(lags, lo, hi, color="0.85", label="95% shuffle null")
    ax.plot(lags, rho, marker="o", ms=3, color="#5b4bdb", label="autocorrelation")
    ax.axhline(0, color="0.6", lw=0.8)
    ax.set_title(sym)
    ax.set_xticks([1, 6, 12, 18, 24])
axes[0, 0].legend(frameon=False, fontsize=8)
fig.supxlabel("lag (hours)")
fig.supylabel("autocorrelation of deseasonalized log volatility")
fig.suptitle("Q1. Volatility clusters: high-volatility hours are followed by high-volatility hours")
fig.tight_layout()
fig.savefig(CHARTS / "q1_volatility_acf.png", bbox_inches="tight")

q1 = pd.DataFrame(q1)
med = q1.lag1.median()
findings.append({"question": "Q1 volatility clustering",
                 "finding": f"median lag-1 autocorrelation {med:.2f} across pairs (per pair {q1.lag1.min():.2f}-{q1.lag1.max():.2f}, "
                            f"95% CIs spanning {q1.ci_lo.min():.2f} to {q1.ci_hi.max():.2f}); {int(q1.beyond_null.sum())}/{len(q1)} pairs above the 95% shuffle null "
                            f"(null upper bound at most {q1.null_hi.max():.2f}, for {q1.loc[q1.null_hi.idxmax(), 'symbol']}; median {q1.null_hi.median():.2f})"})
q1.round(3)

,symbol,lag1,ci_lo,ci_hi,null_hi,beyond_null,lag24,hours
0,ADA,0.711,0.647,0.758,0.045,True,0.413,1974
1,AVAX,0.762,0.688,0.816,0.072,True,0.486,1298
2,BTC,0.786,0.732,0.823,0.042,True,0.353,2161
3,DOGE,0.757,0.699,0.798,0.043,True,0.504,2058
4,ETH,0.746,0.681,0.792,0.039,True,0.346,2161
5,POL,0.615,0.420,0.742,0.122,True,0.175,500
6,SOL,0.765,0.719,0.801,0.041,True,0.473,2161
7,XRP,0.848,0.808,0.877,0.040,True,0.597,2161


## Q2. When in the day and week is each market most active?

**Normalization:** each hour's realized volatility is divided by that pair's overall mean, so 1.0 means "a typical hour for this pair" and all pairs share one scale.

**Test:** Kruskal-Wallis across the 24 UTC hours, per pair. It's rank-based, so volatility's long right tail doesn't dominate. p-values get a **Holm correction**, since 8 pairs are tested.

**Effect size:** the peak hour ÷ the quietest hour, with a CI that resamples whole days. The peak and trough hours are chosen from the full sample, which could bias the point estimate upwards. A split-half check (choose the hours on one half, measure on the other) puts that bias at about 0.04.

**Assumption:** Kruskal-Wallis treats hours as independent, and Q1 shows they aren't, so its p-values are optimistic. Only effects far beyond the threshold should be read as significant.

**Caveat:** 90 days leaves about 13 samples per weekday × hour cell, so the heatmap shows patterns, not precise cell values.

In [3]:
vol["norm"] = vol.realized_vol / vol.groupby("symbol").realized_vol.transform("mean")
heat = vol.pivot_table(index="weekday", columns="hour_utc", values="norm", aggfunc="mean")

fig, ax = plt.subplots(figsize=(13, 3.8))
im = ax.imshow(heat.to_numpy(), aspect="auto", cmap="magma")
ax.set_yticks(range(7), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
ax.set_xticks(range(0, 24, 2), [f"{h:02d}" for h in range(0, 24, 2)])
ax.set_xlabel("hour (UTC)")
fig.colorbar(im, ax=ax, label="volatility vs pair's average (1.0 = typical)")
ax.set_title("Q2. Volatility by weekday and hour, averaged across 8 pairs")
fig.tight_layout()
fig.savefig(CHARTS / "q2_seasonality_heatmap.png", bbox_inches="tight")

by_hour = vol.groupby("hour_utc").norm.mean()
peak, trough = int(by_hour.idxmax()), int(by_hour.idxmin())
vol["day"] = vol.hour_start.dt.date
blocks = [d[["hour_utc", "norm"]].to_numpy() for _, d in vol.groupby("day")]


def peak_trough_ratio(rows):
    h, v = rows[:, 0], rows[:, 1]
    return v[h == peak].mean() / v[h == trough].mean()


ratio_ci = block_bootstrap_ci(blocks, peak_trough_ratio, rng=rng())
weekend = vol.weekday >= 6
wk_ratio = vol[~weekend].norm.mean() / vol[weekend].norm.mean()
wk_ci = block_bootstrap_ci([d[["weekday", "norm"]].to_numpy() for _, d in vol.groupby("day")],
                           lambda r: r[r[:, 0] < 6, 1].mean() / r[r[:, 0] >= 6, 1].mean(), rng=rng())

kw = pd.DataFrame([{"symbol": s, "kruskal_p": sps.kruskal(*[x.norm for _, x in g.groupby("hour_utc")]).pvalue}
                   for s, g in vol.groupby("symbol")])
kw["holm_p"] = holm(kw.kruskal_p)
clear = kw[kw.holm_p < 0.001]
marginal = kw[(kw.holm_p >= 0.001) & (kw.holm_p < 0.05)]

findings.append({"question": "Q2 intraday seasonality",
                 "finding": f"peak hour {peak:02d}:00 UTC is {by_hour.max() / by_hour.min():.2f}x the quietest ({trough:02d}:00 UTC), "
                            f"95% CI {fmt_ci(ratio_ci, 2)}; hour effect clear (Holm p < 0.001) for {len(clear)}/{len(kw)} pairs"
                            + "".join(f", marginal for {r.symbol} (p = {r.holm_p:.3f}, and the test assumes independent hours)" for r in marginal.itertuples())})
findings.append({"question": "Q2 weekday vs weekend",
                 "finding": f"weekday hours are {wk_ratio:.2f}x as volatile as weekend hours, 95% CI {fmt_ci(wk_ci, 2)}"
                            + (" (includes 1: not distinguishable with 13 weeks)" if wk_ci[0] <= 1 <= wk_ci[1] else "")})
print(f"peak {peak:02d}:00 UTC, trough {trough:02d}:00 UTC")
kw.round(6)

peak 14:00 UTC, trough 06:00 UTC


,symbol,kruskal_p,holm_p
0,ADA,0.000000,0.000000
1,AVAX,0.000000,0.000000
2,BTC,0.000000,0.000000
3,DOGE,0.000000,0.000000
4,ETH,0.000000,0.000000
5,POL,0.043789,0.043789
6,SOL,0.000000,0.000000
7,XRP,0.000000,0.000000


## Q3. After an extreme 1-minute move, does the price continue or revert?

**Signal:** a 1-minute return with |z| > 4 against the previous 60 returns, the same rule the Flink detector uses (in SQL, with a rolling window instead of Flink's EWMA).

**Outcome:** "continued" means the 15-minute forward return has the same sign as the move. Minutes with a zero return now or 15 minutes later are excluded on both sides; on coarse-tick pairs they would otherwise count as "reverted".

**Baseline:** the same continuation rate over every scored minute that is **not** a signal. A signal rate only means something relative to it, and excluding the signals keeps the two samples separate for the test.

**Tests:** the difference comes with a CI that resamples whole days. Per pair, a two-proportion z-test with Holm correction; it assumes independent minutes, and about 17% of signals fall within 15 minutes of another, so the pooled bootstrap CI is the more honest number. Follow-through is the mean forward return in the move's direction, in basis points.

In [4]:
mo = q('''
    select symbol, day,
           case when not is_signal then 'baseline'
                when abs(z_score) >= 8 then 'HIGH' when abs(z_score) >= 6 then 'MEDIUM' else 'LOW' end as grp,
           count(*) as n,
           count(*) filter (where continued_15m) as k,
           sum(sign(log_return) * fwd_return_15m) as signed_fwd
    from analytics.mart_minute_outcomes
    group by 1, 2, 3
''')
mo["signal"] = mo.grp != "baseline"


def day_blocks(df, groups):
    # one block per day: rows of [k, n, signed_fwd] for each group, columns aligned
    wide = df.pivot_table(index="day", columns="grp", values=["k", "n", "signed_fwd"], aggfunc="sum", fill_value=0)
    return [np.array([[wide.loc[d, (m, g)] if (m, g) in wide.columns else 0 for m in ("k", "n", "signed_fwd")] for g in groups])[None]
            for d in wide.index]


def rate(rows, i):
    return rows[:, i, 0].sum() / rows[:, i, 1].sum()


groups = ["baseline", "LOW", "MEDIUM", "HIGH"]
pooled = mo.assign(grp=np.where(mo.signal, "signal", "baseline"))
blocks_all = day_blocks(pooled, ["baseline", "signal"])
blocks_sev = day_blocks(mo, groups)

tot = pooled.groupby("grp")[["k", "n", "signed_fwd"]].sum()
base_rate, sig_rate = tot.loc["baseline", "k"] / tot.loc["baseline", "n"], tot.loc["signal", "k"] / tot.loc["signal", "n"]
diff_ci = block_bootstrap_ci(blocks_all, lambda r: rate(r, 1) - rate(r, 0), rng=rng())
base_ci = block_bootstrap_ci(blocks_all, lambda r: rate(r, 0), rng=rng())
sig_ci = block_bootstrap_ci(blocks_all, lambda r: rate(r, 1), rng=rng())
z, p = two_proportion_z(tot.loc["signal", "k"], tot.loc["signal", "n"], tot.loc["baseline", "k"], tot.loc["baseline", "n"])
bps = tot.loc["signal", "signed_fwd"] / tot.loc["signal", "n"] * 1e4
bps_ci = block_bootstrap_ci(blocks_all, lambda r: r[:, 1, 2].sum() / r[:, 1, 1].sum() * 1e4, rng=rng())
base_bps = tot.loc["baseline", "signed_fwd"] / tot.loc["baseline", "n"] * 1e4

sev = []
for i, g in enumerate(groups):
    k, n = mo[mo.grp == g].k.sum(), mo[mo.grp == g].n.sum()
    lo, hi = block_bootstrap_ci(blocks_sev, lambda r, i=i: rate(r, i), rng=rng()) if n else (np.nan, np.nan)
    sev.append({"group": g, "n": int(n), "continued": k / n if n else np.nan, "ci_lo": lo, "ci_hi": hi})
sev = pd.DataFrame(sev)

per_pair = []
for s, g in mo.groupby("symbol"):
    t = g.groupby("signal")[["k", "n"]].sum()
    if True in t.index and t.loc[True, "n"] > 0:
        zz, pp = two_proportion_z(t.loc[True, "k"], t.loc[True, "n"], t.loc[False, "k"], t.loc[False, "n"])
        per_pair.append({"symbol": s, "signals": int(t.loc[True, "n"]), "signal_rate": t.loc[True, "k"] / t.loc[True, "n"],
                         "baseline_rate": t.loc[False, "k"] / t.loc[False, "n"], "p": pp})
per_pair = pd.DataFrame(per_pair)
per_pair["holm_p"] = holm(per_pair.p)

fig, ax = plt.subplots(figsize=(7.5, 4))
colors = ["0.45", "#8fb8ff", "#5b4bdb", "#2a1f8f"]
labels = ["non-signal\nminutes", "LOW\n|z| 4-6", "MEDIUM\n|z| 6-8", "HIGH\n|z| >= 8"]
for x_, r in enumerate(sev.itertuples()):  # dot + interval, so the axis needn't start at 0
    ax.errorbar(x_, r.continued, yerr=[[r.continued - r.ci_lo], [r.ci_hi - r.continued]], fmt="o", ms=8, color=colors[x_], capsize=5)
    ax.annotate(f"n={r.n:,}", (x_, r.ci_lo), textcoords="offset points", xytext=(0, -14), ha="center", fontsize=8)
ax.axhline(sev.continued.iloc[0], color="0.45", ls=":", lw=0.8)
ax.axhline(0.5, color="0.7", ls="--", lw=0.8)
ax.set_xticks(range(len(labels)), labels)
ax.set_xlim(-0.5, len(labels) - 0.5)
ax.set_ylim(sev.ci_lo.min() - 0.03, sev.ci_hi.max() + 0.01)  # room for the n= labels
ax.set_ylabel("share that continued 15 min later")
ax.set_title("Q3. Extreme moves continue less often than ordinary minutes\n(pooled; 95% CIs resample whole days, HIGH overlaps the baseline)")
fig.tight_layout()
fig.savefig(CHARTS / "q3_continuation.png", bbox_inches="tight")

findings.append({"question": "Q3 continuation vs baseline",
                 "finding": f"{tot.loc['signal', 'n']:,} signals continued {sig_rate:.1%} [{sig_ci[0]:.1%}, {sig_ci[1]:.1%}] vs {base_rate:.1%} [{base_ci[0]:.1%}, {base_ci[1]:.1%}] for the {tot.loc['baseline', 'n']:,} non-signal minutes; "
                            f"difference {(sig_rate - base_rate) * 100:+.1f} pts, 95% CI [{diff_ci[0] * 100:+.1f}, {diff_ci[1] * 100:+.1f}] (day-block bootstrap); "
                            f"per pair, significant in {int((per_pair.holm_p < 0.05).sum())}/{len(per_pair)} (two-proportion z, Holm; assumes independent minutes)"})
findings.append({"question": "Q3 size of the follow-through",
                 "finding": f"mean 15-min return in the signal's direction {bps:+.1f} bps, 95% CI [{bps_ci[0]:+.1f}, {bps_ci[1]:+.1f}] (ordinary minutes {base_bps:+.2f} bps)"
                            + ("; not distinguishable from zero" if bps_ci[0] <= 0 <= bps_ci[1] else "")})
sev.round(4), per_pair.round(4)

(      group       n  continued   ci_lo   ci_hi
 0  baseline  652357     0.4946  0.4926  0.4966
 1       LOW    3004     0.4521  0.4297  0.4744
 2    MEDIUM     500     0.4440  0.3988  0.4881
 3      HIGH     352     0.4659  0.3994  0.5325,
   symbol  signals  signal_rate  baseline_rate       p  holm_p
 0    ADA      341       0.4839         0.4969  0.6302  0.8470
 1   AVAX      258       0.5271         0.5021  0.4235  0.8470
 2    BTC      797       0.4417         0.4930  0.0038  0.0268
 3   DOGE      374       0.4492         0.4991  0.0544  0.2720
 4    ETH      765       0.4431         0.4933  0.0056  0.0337
 5    POL       76       0.4211         0.5099  0.1233  0.3700
 6    SOL      608       0.4556         0.4903  0.0880  0.3519
 7    XRP      637       0.4317         0.4948  0.0015  0.0120)

## Q4. How closely does the streaming pipeline agree with the exchange?

**Data:** minutes the stream built by itself (repaired minutes are excluded, since they were rebuilt from Coinbase's own trade record), compared with Coinbase's official candle for the same minute.

**Agreement:** the close is within 0.05% and the volume within 5%. Wilson 95% intervals, since each pair has only about 130–180 minutes.

**Hypothesis tested:** close-price misses on cheap pairs are one-tick differences. The decisive check is whether one tick can exceed the 0.05% tolerance at all, even at each pair's lowest price in the 90 days. A Spearman rank correlation across the 8 pairs is shown too, but with n = 8 it's underpowered and only secondary.

In [5]:
agr = q('''
    select s.symbol, s.n_minutes, s.pct_close_agree, s.pct_volume_agree, s.pct_agree, t.quote_increment
    from analytics.mart_pipeline_vs_exchange_summary s
    join analytics.product_ticks t using (symbol)
''')
# anchored on the data, not the clock, so a re-run later gives the same numbers
px = q("select symbol, percentile_cont(0.5) within group (order by close) as median_close, min(close) as min_close from analytics.fct_candles_1m group by 1")
agr = agr.merge(px, on="symbol")
agr["tick_pct"] = agr.quote_increment.astype(float) / agr.median_close.astype(float) * 100
agr["tick_pct_at_low"] = agr.quote_increment.astype(float) / agr.min_close.astype(float) * 100  # worst case over 90 days
for col in ("close", "volume"):
    k = (agr[f"pct_{col}_agree"].astype(float) * agr.n_minutes).round()
    cis = [wilson_ci(kk, nn) for kk, nn in zip(k, agr.n_minutes)]
    agr[f"{col}_lo"], agr[f"{col}_hi"] = zip(*cis)
rho, rho_p = sps.spearmanr(agr.tick_pct, agr.pct_close_agree.astype(float))

fig, ax = plt.subplots(figsize=(7.5, 4.2))
y = agr.pct_close_agree.astype(float)
ax.errorbar(agr.tick_pct, y, yerr=[y - agr.close_lo, agr.close_hi - y], fmt="o", color="#5b4bdb", capsize=3)
for _, r in agr.iterrows():
    ax.annotate(r.symbol, (r.tick_pct, float(r.pct_close_agree)), textcoords="offset points", xytext=(6, 3), fontsize=8)
ax.axvline(0.05, color="0.5", ls="--", lw=0.8)
ax.text(0.05, 0.02, " tolerance 0.05%", transform=ax.get_xaxis_transform(), fontsize=8, color="0.4")
ax.set_xscale("log")
ax.set_xlabel("one price tick as % of price (log scale)")
ax.set_ylabel("minutes where close agrees (Wilson 95% CI)")
ax.set_title(f"Q4. Every tick is far below the 0.05% tolerance, so ticks don't explain the misses (rho = {rho:.2f}, p = {rho_p:.2f})")
fig.tight_layout()
fig.savefig(CHARTS / "q4_agreement_ticks.png", bbox_inches="tight")

majors = agr[agr.symbol.isin(["BTC", "ETH", "SOL", "XRP"])]
findings.append({"question": "Q4 pipeline vs exchange",
                 "finding": f"BTC/ETH/SOL/XRP: close agrees on {majors.pct_close_agree.astype(float).min():.1%}-{majors.pct_close_agree.astype(float).max():.1%} of minutes, "
                            f"volume on {majors.pct_volume_agree.astype(float).min():.1%}-{majors.pct_volume_agree.astype(float).max():.1%} "
                            f"(about {int(majors.n_minutes.min())}-{int(majors.n_minutes.max())} minutes each); "
                            f"tick-size explanation ruled out: even at its 90-day low price the largest tick is {agr.tick_pct_at_low.max():.3f}% of price ({agr.loc[agr.tick_pct_at_low.idxmax(), 'symbol']}), "
                            f"below the 0.05% tolerance, so a one-tick difference can't fail the check (rank correlation rho = {rho:.2f}, p={rho_p:.2g}, n = 8, is underpowered and secondary); "
                            f"POL's close misses ({1 - float(agr.set_index('symbol').loc['POL', 'pct_close_agree']):.0%}) remain unexplained"})
agr[["symbol", "n_minutes", "tick_pct", "tick_pct_at_low", "pct_close_agree", "close_lo", "close_hi", "pct_volume_agree", "volume_lo", "volume_hi"]].round(4)

,symbol,n_minutes,tick_pct,tick_pct_at_low,pct_close_agree,close_lo,close_hi,pct_volume_agree,volume_lo,volume_hi
0,ADA,180,0.0052,0.0070,0.9667,0.9292,0.9846,0.7111,0.6410,0.7724
1,AVAX,183,0.0138,0.0165,0.9727,0.9376,0.9883,0.8033,0.7397,0.8544
2,BTC,184,0.0000,0.0000,1.0000,0.9795,1.0000,0.9565,0.9166,0.9778
3,DOGE,172,0.0134,0.0147,0.9186,0.8680,0.9509,0.6279,0.5536,0.6966
4,ETH,184,0.0005,0.0006,0.9946,0.9699,0.9990,0.9022,0.8507,0.9372
5,POL,132,0.0105,0.0148,0.6894,0.6060,0.7620,0.4621,0.3793,0.5470
6,SOL,183,0.0127,0.0142,1.0000,0.9794,1.0000,0.9071,0.8563,0.9412
7,XRP,183,0.0090,0.0101,1.0000,0.9794,1.0000,0.9344,0.8889,0.9621


## Limitations

- **Dependence across days.** The CIs resample whole days, but Q1 shows volatility persists for days (lag-24 autocorrelation 0.17–0.60), so day-block CIs are slightly narrow. Re-running with week blocks moves no conclusion across zero or one.
- **Independence assumptions.** The Kruskal-Wallis (Q2) and per-pair z-tests (Q3) assume independent hours or minutes. Their p-values are optimistic, which is why the pooled bootstrap CIs are the headline numbers.
- **One exchange, 90 days.** This describes Coinbase from June to September 2026, not crypto markets in general.

## Findings

These are the numbers quoted in `analysis/README.md`, printed by this cell and not typed by hand.

In [6]:
conn.close()
for f in findings:
    print(f"- {f['question']}: {f['finding']}")

- Q1 volatility clustering: median lag-1 autocorrelation 0.76 across pairs (per pair 0.61-0.85, 95% CIs spanning 0.42 to 0.88); 8/8 pairs above the 95% shuffle null (null upper bound at most 0.12, for POL; median 0.04)
- Q2 intraday seasonality: peak hour 14:00 UTC is 1.80x the quietest (06:00 UTC), 95% CI [1.60, 1.99]; hour effect clear (Holm p < 0.001) for 7/8 pairs, marginal for POL (p = 0.044, and the test assumes independent hours)
- Q2 weekday vs weekend: weekday hours are 1.26x as volatile as weekend hours, 95% CI [0.98, 1.61] (includes 1: not distinguishable with 13 weeks)
- Q3 continuation vs baseline: 3,856 signals continued 45.2% [42.9%, 47.4%] vs 49.5% [49.3%, 49.7%] for the 652,357 non-signal minutes; difference -4.2 pts, 95% CI [-6.5, -2.0] (day-block bootstrap); per pair, significant in 3/8 (two-proportion z, Holm; assumes independent minutes)
- Q3 size of the follow-through: mean 15-min return in the signal's direction +2.1 bps, 95% CI [-1.6, +6.5] (ordinary minutes +0.